# Inspect ResNet293 Model

This notebook loads the WeSpeaker ResNet293 PyTorch checkpoint, runs one audio sample through it, and shows how tensor shapes change across the main layers.


In [1]:
import sys
import types
from pathlib import Path
import importlib.util

import pandas as pd
import soundfile as sf
import torch
import torchaudio
import torchaudio.compliance.kaldi as kaldi
import yaml


In [2]:
# ===== Config =====
REPO_ROOT = Path('/home/SpeakerRec/BioVoice')
MODEL_DIR = REPO_ROOT / 'data' / 'models' / 'wespeaker_resnet293_lm'
AUDIO_PATH = REPO_ROOT / 'data' / 'wavs' / 'eden_001.wav'
WESPEAKER_MODELS_DIR = REPO_ROOT / 'external' / 'wespeaker' / 'wespeaker' / 'models'

CONFIG_PATH = MODEL_DIR / 'config.yaml'
CKPT_PATH = MODEL_DIR / 'avg_model.pt'
TARGET_SR = 16000

print('CONFIG EXISTS =', CONFIG_PATH.exists())
print('CKPT EXISTS =', CKPT_PATH.exists())
print('AUDIO EXISTS =', AUDIO_PATH.exists())
print('WESPEAKER_MODELS_DIR EXISTS =', WESPEAKER_MODELS_DIR.exists())


CONFIG EXISTS = True
CKPT EXISTS = True
AUDIO EXISTS = True
WESPEAKER_MODELS_DIR EXISTS = True


In [3]:
# Load only the exact WeSpeaker model files we need.
wespeaker_pkg = types.ModuleType('wespeaker')
wespeaker_models_pkg = types.ModuleType('wespeaker.models')
sys.modules['wespeaker'] = wespeaker_pkg
sys.modules['wespeaker.models'] = wespeaker_models_pkg

pool_spec = importlib.util.spec_from_file_location(
    'wespeaker.models.pooling_layers',
    str(WESPEAKER_MODELS_DIR / 'pooling_layers.py'),
)
pool_mod = importlib.util.module_from_spec(pool_spec)
sys.modules['wespeaker.models.pooling_layers'] = pool_mod
pool_spec.loader.exec_module(pool_mod)

resnet_spec = importlib.util.spec_from_file_location(
    'wespeaker.models.resnet',
    str(WESPEAKER_MODELS_DIR / 'resnet.py'),
)
resnet_mod = importlib.util.module_from_spec(resnet_spec)
sys.modules['wespeaker.models.resnet'] = resnet_mod
resnet_spec.loader.exec_module(resnet_mod)

ResNet293 = resnet_mod.ResNet293
print('ResNet293 constructor loaded.')


ResNet293 constructor loaded.


In [4]:
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    configs = yaml.safe_load(f)

print('model =', configs['model'])
print('model_args =', configs['model_args'])
print('fbank_args =', configs['dataset_args']['fbank_args'])

model = ResNet293(**configs['model_args'])
state = torch.load(CKPT_PATH, map_location='cpu')

if isinstance(state, dict):
    if 'state_dict' in state:
        state = state['state_dict']
    elif 'model' in state:
        state = state['model']

if isinstance(state, dict):
    state = { (k[7:] if k.startswith('module.') else k): v for k, v in state.items() }

missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()

print('Missing keys:', missing)
print('Unexpected keys:', unexpected)
model


model = ResNet293
model_args = {'embed_dim': 256, 'feat_dim': 80, 'pooling_func': 'TSTP', 'two_emb_layer': False}
fbank_args = {'dither': 1.0, 'frame_length': 25, 'frame_shift': 10, 'num_mel_bins': 80}


/tmp/ipykernel_1029657/444115592.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(CKPT_PATH, map_location='cpu')


Missing keys: []
Unexpected keys: ['projection.weight']


ResNet(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(32, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential(
        (0): Conv2d(32, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (1): Bottleneck(
      (

In [5]:
waveform_np, sr = sf.read(str(AUDIO_PATH))
print('loaded waveform raw shape =', waveform_np.shape, '| sr =', sr)

if waveform_np.ndim == 1:
    waveform_np = waveform_np[None, :]
else:
    waveform_np = waveform_np.T

waveform = torch.from_numpy(waveform_np).float()
if sr != TARGET_SR:
    waveform = torchaudio.functional.resample(waveform, sr, TARGET_SR)
    sr = TARGET_SR

waveform = waveform[:1, :]
waveform = waveform * (1 << 15)

fbank_args = configs['dataset_args']['fbank_args']
feats = kaldi.fbank(
    waveform,
    num_mel_bins=fbank_args['num_mel_bins'],
    frame_length=fbank_args['frame_length'],
    frame_shift=fbank_args['frame_shift'],
    dither=0.0,
    sample_frequency=TARGET_SR,
)

feats = feats.unsqueeze(0)
print('input feature tensor shape =', tuple(feats.shape))
display(feats[0, :5, :8])


loaded waveform raw shape = (37797,) | sr = 16000
input feature tensor shape = (1, 234, 80)


tensor([[ 5.6752,  6.0484,  6.3748,  6.9200,  7.9883,  8.5381,  8.4804,  8.0205],
        [ 7.2703,  8.7437,  9.3083,  8.5273,  8.2263,  8.4279,  8.8917,  8.8856],
        [ 7.2137,  8.8608, 10.1529,  9.8497,  9.3259,  8.1394,  8.7918,  8.6659],
        [ 7.5423,  8.9733,  9.6813,  8.9232,  8.0938,  6.7217,  7.8854,  8.4383],
        [ 7.4685,  8.6832,  9.1158,  8.1856,  7.5254,  7.0332,  6.2920,  4.4959]])

In [6]:
shape_rows = []
hooks = []

def _shape_of(x):
    if isinstance(x, torch.Tensor):
        return tuple(x.shape)
    if isinstance(x, (list, tuple)):
        return [_shape_of(v) for v in x]
    return str(type(x))

def register_shape_hook(module_name, module):
    def hook(_module, inputs, outputs):
        shape_rows.append({
            'layer': module_name,
            'input_shape': _shape_of(inputs[0] if len(inputs) == 1 else inputs),
            'output_shape': _shape_of(outputs),
        })
    hooks.append(module.register_forward_hook(hook))

modules_to_trace = [
    ('conv1', model.conv1),
    ('bn1', model.bn1),
    ('layer1', model.layer1),
    ('layer2', model.layer2),
    ('layer3', model.layer3),
    ('layer4', model.layer4),
    ('pool', model.pool),
    ('seg_1', model.seg_1),
]

for name, module in modules_to_trace:
    register_shape_hook(name, module)

with torch.no_grad():
    outputs = model(feats)
    emb = outputs[-1] if isinstance(outputs, tuple) else outputs

for h in hooks:
    h.remove()

shape_df = pd.DataFrame(shape_rows)
display(shape_df)

print('final embedding shape =', tuple(emb.shape))
print('first 10 values =', emb[0, :10])


,layer,input_shape,output_shape
0,conv1,"(1, 1, 80, 234)","(1, 32, 80, 234)"
1,bn1,"(1, 32, 80, 234)","(1, 32, 80, 234)"
2,layer1,"(1, 32, 80, 234)","(1, 128, 80, 234)"
3,layer2,"(1, 128, 80, 234)","(1, 256, 40, 117)"
4,layer3,"(1, 256, 40, 117)","(1, 512, 20, 59)"
5,layer4,"(1, 512, 20, 59)","(1, 1024, 10, 30)"
6,pool,"(1, 1024, 10, 30)","(1, 20480)"
7,seg_1,"(1, 20480)","(1, 256)"


final embedding shape = (1, 256)
first 10 values = tensor([ 0.0172, -0.1383,  0.1308,  0.0730,  0.1149,  0.1743, -0.0011, -0.2975,
         0.2450, -0.2618])


In [7]:
# Inspect the helper methods around the convolution stack.
with torch.no_grad():
    frame_level_4d = model._get_frame_level_feat(feats)
    frame_level_flat = model.get_frame_level_feat(feats)
    pooled = model.pool(frame_level_4d)
    seg1 = model.seg_1(pooled)

print('input feats [B,T,F]        =', tuple(feats.shape))
print('_get_frame_level_feat      =', tuple(frame_level_4d.shape))
print('get_frame_level_feat       =', tuple(frame_level_flat.shape))
print('pool output               =', tuple(pooled.shape))
print('seg_1 output              =', tuple(seg1.shape))


input feats [B,T,F]        = (1, 234, 80)
_get_frame_level_feat      = (1, 1024, 10, 30)
get_frame_level_feat       = (1, 30, 10240)
pool output               = (1, 20480)
seg_1 output              = (1, 256)


In [8]:
# Optional deeper look: shapes of the first block in each residual stage.
block_rows = []
block_hooks = []

for block_name in ['layer1.0', 'layer2.0', 'layer3.0', 'layer4.0']:
    module = dict(model.named_modules())[block_name]
    def _make_hook(name):
        def hook(_module, inputs, outputs):
            block_rows.append({
                'block': name,
                'input_shape': tuple(inputs[0].shape),
                'output_shape': tuple(outputs.shape),
            })
        return hook
    block_hooks.append(module.register_forward_hook(_make_hook(block_name)))

with torch.no_grad():
    _ = model(feats)

for h in block_hooks:
    h.remove()

display(pd.DataFrame(block_rows))


,block,input_shape,output_shape
0,layer1.0,"(1, 32, 80, 234)","(1, 128, 80, 234)"
1,layer2.0,"(1, 128, 80, 234)","(1, 256, 40, 117)"
2,layer3.0,"(1, 256, 40, 117)","(1, 512, 20, 59)"
3,layer4.0,"(1, 512, 20, 59)","(1, 1024, 10, 30)"
